<a href="https://colab.research.google.com/github/ck1972/University-GeoAI/blob/main/Mod2_Lab3d_Building_Segmentation_PreTrained_UNET_Part1_GitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Semantic Segmentation of Aerial Imagery for Building Footprint Extraction Using Transfer Learning with a Pretrained U-Net (with ResNet34 Encoder)**

## Install required libraries

In [ ]:
# Install segmentation_models_pytorch: for U-Net and other deep learning segmentation architectures
!pip install segmentation-models-pytorch --quiet

# Install albumentations: for efficient image augmentation and preprocessing
!pip install albumentations --quiet

# Install buildingregulariser: for refining and regularizing predicted building footprints
!pip install buildingregulariser --quiet

# Install rasterio: for reading and writing raster data such as satellite or aerial imagery
!pip install rasterio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 107.7 MB/s eta 0:00:00


## Import required libraries

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterio import features
from shapely.geometry import shape
from rasterio.features import shapes
from buildingregulariser import regularize_geodataframe
from sklearn.metrics import f1_score, jaccard_score
from scipy.ndimage import binary_dilation, binary_erosion
import torch
from torch.utils.data import Dataset
from torch.utils.data import random_split, DataLoader
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random

## Mount Drive and load inputs

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/')
os.chdir('/content/drive/MyDrive/Chit_TR9207')

Mounted at /content/drive/


## Load image and rasterize GeoJSON to create a mask

In [ ]:
# Load raster and geojson

# Load image
img_path = 'TR9207a.tif'
with rasterio.open(img_path) as src:
    image = src.read([1, 2, 3])  # Assume RGB bands
    transform = src.transform
    height, width = src.height, src.width
    crs = src.crs

# Load GeoJSON
gdf = gpd.read_file('Bld_DSG_TR9207.geojson')

# Rasterize buildings
mask = features.rasterize(
    ((geom, 1) for geom in gdf.geometry),
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=np.uint8
)

## Create patches from image and mask

In [ ]:
# Create patacjes from image and mask
class PatchDataset(Dataset):
    def __init__(self, image, mask, patch_size=256):
        self.image = image
        self.mask = mask
        self.patch_size = patch_size
        self.coords = [
            (i, j)
            for i in range(0, image.shape[1] - patch_size + 1, patch_size)
            for j in range(0, image.shape[2] - patch_size + 1, patch_size)
        ]

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        i, j = self.coords[idx]
        img_patch = self.image[:, i:i+self.patch_size, j:j+self.patch_size]
        mask_patch = self.mask[i:i+self.patch_size, j:j+self.patch_size]
        return torch.tensor(img_patch / 255.0, dtype=torch.float32), torch.tensor(mask_patch, dtype=torch.long)

## Split data and create dataLoaders

In [ ]:
# Spit data and create dataloaders
dataset = PatchDataset(image, mask)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

## Load pretrained U-Net (transfer learning)

In [ ]:
# Load pretrained U-Net model
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
).to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

## Train the model (binary crossentropy)

In [ ]:
# Train the model
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(10):
    model.train()
    total_loss = 0
    correct, total = 0, 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device).float().unsqueeze(1)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds_binary = (torch.sigmoid(preds) > 0.5).float()
        correct += (preds_binary == masks).sum().item()
        total += torch.numel(masks)

    train_losses.append(total_loss / len(train_loader))
    train_accs.append(correct / total)

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device).float().unsqueeze(1)
            preds = model(imgs)
            val_loss += criterion(preds, masks).item()
            preds_binary = (torch.sigmoid(preds) > 0.5).float()
            correct += (preds_binary == masks).sum().item()
            total += torch.numel(masks)

    val_losses.append(val_loss / len(val_loader))
    val_accs.append(correct / total)

    print(f"Epoch {epoch+1}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, "
          f"Train Acc: {train_accs[-1]:.4f}, Val Acc: {val_accs[-1]:.4f}")

Epoch 1, Train Loss: 0.0542, Val Loss: 0.0741, Train Acc: 0.9790, Val Acc: 0.9733
Epoch 2, Train Loss: 0.0470, Val Loss: 0.0737, Train Acc: 0.9815, Val Acc: 0.9747
Epoch 3, Train Loss: 0.0466, Val Loss: 0.0744, Train Acc: 0.9816, Val Acc: 0.9741
Epoch 4, Train Loss: 0.0418, Val Loss: 0.0777, Train Acc: 0.9834, Val Acc: 0.9747
Epoch 5, Train Loss: 0.0377, Val Loss: 0.0809, Train Acc: 0.9849, Val Acc: 0.9749
Epoch 6, Train Loss: 0.0405, Val Loss: 0.0750, Train Acc: 0.9837, Val Acc: 0.9742
Epoch 7, Train Loss: 0.0387, Val Loss: 0.0769, Train Acc: 0.9843, Val Acc: 0.9750
Epoch 8, Train Loss: 0.0387, Val Loss: 0.0789, Train Acc: 0.9844, Val Acc: 0.9723
Epoch 9, Train Loss: 0.0437, Val Loss: 0.0780, Train Acc: 0.9826, Val Acc: 0.9738
Epoch 10, Train Loss: 0.0411, Val Loss: 0.0815, Train Acc: 0.9836, Val Acc: 0.9751


## Visualize training vs. validation loss and accuracy

To assess the model's learning progress, plot the training and validation loss and accuracy over epochs:

In [ ]:
# Plot Loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## Predict full image mask

In [ ]:
# Predict full image mask
model.eval()
pred_mask = np.zeros((height, width), dtype=np.uint8)

with torch.no_grad():
    for i in range(0, height - 256 + 1, 256):
        for j in range(0, width - 256 + 1, 256):
            patch = image[:, i:i+256, j:j+256]
            patch_tensor = torch.tensor(patch/255.0, dtype=torch.float32).unsqueeze(0).to(device)
            output = model(patch_tensor)
            output = torch.sigmoid(output)
            pred = (output.squeeze().cpu().numpy() > 0.5).astype(np.uint8)
            pred_mask[i:i+256, j:j+256] = pred

## Evaluate full-image segmentation with standard metrics

In [ ]:
# Evaluate full-image segmentation with standard metrics
from sklearn.metrics import f1_score, jaccard_score
from scipy.ndimage import binary_dilation, binary_erosion

# Flattened metrics
gt_flat = mask.flatten()
pred_flat = pred_mask.flatten()

# F1, IoU, Dice
f1 = f1_score(gt_flat, pred_flat)
iou = jaccard_score(gt_flat, pred_flat)
dice = 2 * (iou * f1) / (iou + f1)

# Compute Boundary Masks
def extract_boundary(mask, dilation_size=2):
    """Extract boundary of binary mask using morphological gradient (dilate - erode)."""
    dilated = binary_dilation(mask, iterations=dilation_size)
    eroded = binary_erosion(mask, iterations=dilation_size)
    boundary = np.logical_xor(dilated, eroded).astype(np.uint8)
    return boundary

gt_boundary = extract_boundary(mask)
pred_boundary = extract_boundary(pred_mask)

# Compute Boundary IoU
intersection = np.logical_and(gt_boundary, pred_boundary).sum()
union = np.logical_or(gt_boundary, pred_boundary).sum()
biou = intersection / union if union > 0 else 0

# Print metrics
print(f"F1 Score:         {f1:.4f}")
print(f"IoU:              {iou:.4f}")
print(f"Dice Coefficient: {dice:.4f}")
print(f"Boundary IoU:     {biou:.4f}")

F1 Score:         0.9484
IoU:              0.9019
Dice Coefficient: 0.9246
Boundary IoU:     0.5343


## Display sample test patches

In [ ]:
# Select 3 random samples from the validation set
for _ in range(3):
    idx = random.randint(0, len(val_ds) - 1)
    img, gt = val_ds[idx]
    img_tensor = img.unsqueeze(0).to(device)

    # Predict the mask
    with torch.no_grad():
        pred = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()
    pred_binary = (pred > 0.5).astype(np.uint8)

    # Plot the results
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img.permute(1, 2, 0))
    plt.title("Input Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt.numpy(), cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pred_binary, cmap='gray')
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

## Convert mask to polygons and save GeoJSON

In [ ]:
# Convert building masks to polygons
results = (
    {'properties': {'value': v}, 'geometry': s}
    for s, v in shapes(pred_mask, transform=transform)
    if v == 1
)

gdf_pred = gpd.GeoDataFrame.from_features(results, crs=crs)
gdf_pred.to_file("pred_pretrained_UNET_bld9207.geojson", driver="GeoJSON")

## Regularize predicted building polygons

In [ ]:
# Vectorize the predicted mask
results = (
    {"properties": {"raster_val": v}, "geometry": s}
    for s, v in shapes(pred_mask, mask=pred_mask.astype(bool), transform=transform)
)

# Create a GeoDataFrame
gdf_pred = gpd.GeoDataFrame.from_features(results, crs=crs)

# Regularize the building footprints
gdf_reg = regularize_geodataframe(gdf_pred)

# Save the regularized footprints to a GeoJSON file
gdf_reg.to_file("pred_pretrained_UNET_bld9207_regularized.geojson", driver="GeoJSON")

## Save the trained model (weights only)

In [ ]:
# Save model weights
model_path = "pretrained_unet_building_segmentation.pth"  # ✅ clear and descriptive name
torch.save(model.state_dict(), model_path)                # ✅ saves only the weights
print(f"Model weights saved to: {model_path}")            # ✅ confirmation output

Model weights saved to: pretrained_unet_building_segmentation.pth
